In [ ]:
import requests, json

PARAMS = "T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,PS,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,CLOUD_AMT,ALLSKY_SFC_UV_INDEX"

url = "https://power.larc.nasa.gov/api/temporal/climatology/point"
args = {"parameters": PARAMS, "community": "AG",
        "latitude": -23.5, "longitude": -69.5,
        "format": "JSON"}

r = requests.get(url, params=args, timeout=60)
print("Status:", r.status_code)
data = r.json()

if r.status_code != 200:
    print(json.dumps(data, indent=2)[:2000])
else:
    print("Elevation (m):", data["geometry"]["coordinates"][2])
    for name, months in data["properties"]["parameter"].items():
        print(f"{name:22s} JAN={months.get('JAN')}  JUL={months.get('JUL')}  YEAR={months.get('ANN')}")


In [ ]:
!pip -q install global-land-mask
import numpy as np, requests, json, time
from global_land_mask import globe
from concurrent.futures import ThreadPoolExecutor, as_completed

URL = "https://power.larc.nasa.gov/api/temporal/climatology/point"
MONTHS = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]

STEP = 4
points = []
for lat in np.arange(-86, 84, STEP):
    for lon in np.arange(-178, 180, STEP):
        if globe.is_land(lat, lon):
            points.append({"kind": "grid", "name": None, "lat": float(lat), "lon": float(lon)})

KNOWN = [
    ("Atacama Desert (Yungay), Chile", -24.08, -70.01),
    ("McMurdo Dry Valleys, Antarctica", -77.50, 162.00),
    ("Haughton Crater, Devon Island, Canada", 75.38, -89.68),
    ("Death Valley, USA", 36.46, -116.87),
    ("Mars Desert Research Station, Utah, USA", 38.41, -110.79),
    ("Mauna Kea, Hawaii, USA", 19.82, -155.47),
    ("Holuhraun lava field, Iceland", 64.87, -16.83),
    ("Bockfjorden, Svalbard, Norway", 79.40, 13.40),
]
for name, lat, lon in KNOWN:
    points.append({"kind": "known", "name": name, "lat": lat, "lon": lon})
print("Points to fetch:", len(points))

def fetch(p):
    args = {"parameters": PARAMS, "community": "AG",
            "latitude": p["lat"], "longitude": p["lon"], "format": "JSON"}
    for attempt in range(4):
        try:
            r = requests.get(URL, params=args, timeout=60)
            if r.status_code == 200:
                d = r.json()
                out = dict(p)
                out["elev"] = round(d["geometry"]["coordinates"][2], 1)
                out["m"] = {k: [v[mo] for mo in MONTHS]
                            for k, v in d["properties"]["parameter"].items()}
                return out
        except Exception:
            pass
        time.sleep(2 + attempt * 3)
    return None

results, failed = [], 0
t0 = time.time()
with ThreadPoolExecutor(max_workers=5) as ex:
    futures = [ex.submit(fetch, p) for p in points]
    for i, f in enumerate(as_completed(futures), 1):
        res = f.result()
        if res: results.append(res)
        else: failed += 1
        if i % 100 == 0:
            print(f"{i}/{len(points)} done, {time.time()-t0:.0f} s")

print("OK:", len(results), " failed:", failed)
with open("earth_points.json", "w") as fh:
    json.dump(results, fh)
print("Saved earth_points.json")

In [ ]:
def fetch(p):
    args = {"parameters": PARAMS, "community": "AG",
            "latitude": p["lat"], "longitude": p["lon"], "format": "JSON"}
    for attempt in range(3):
        try:
            r = requests.get(URL, params=args, timeout=20)
            if r.status_code == 200:
                d = r.json()
                out = dict(p)
                out["elev"] = round(d["geometry"]["coordinates"][2], 1)
                out["m"] = {k: [v[mo] for mo in MONTHS]
                            for k, v in d["properties"]["parameter"].items()}
                return out
        except Exception:
            pass
        time.sleep(3)
    return None

have = {(r["lat"], r["lon"]) for r in results}
missing = [p for p in points if (p["lat"], p["lon"]) not in have]
print("Already have:", len(results), "| missing:", len(missing))

t0 = time.time()
still_missing = []
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(fetch, p): p for p in missing}
    for i, f in enumerate(as_completed(futures), 1):
        res = f.result()
        if res: results.append(res)
        else: still_missing.append(futures[f])
        if i % 50 == 0:
            print(f"{i}/{len(missing)} done, {time.time()-t0:.0f} s")

print("TOTAL OK:", len(results), "| still missing:", len(still_missing))
with open("earth_points.json", "w") as fh:
    json.dump(results, fh)
print("Saved earth_points.json")

In [ ]:
import pandas as pd

rows = []
for pid, r in enumerate(results):
    for mi in range(12):
        row = {"pid": pid, "kind": r["kind"], "name": r["name"],
               "lat": r["lat"], "lon": r["lon"], "elev": r["elev"], "month": mi}
        for k, vals in r["m"].items():
            v = vals[mi]
            row[k] = None if (v is None or v <= -998) else v   # -999 = no data
        rows.append(row)
df = pd.DataFrame(rows)

FACTORS = {
    "dryness":    [("PRECTOTCORR", "low"), ("RH2M", "low")],
    "cold":       [("T2M", "low")],
    "cold_min":   [("T2M_MIN", "low")],
    "thin_air":   [("PS", "low")],
    "uv":         [("ALLSKY_SFC_UV_INDEX", "high")],
    "temp_swing": [("T2M_RANGE", "high")],
    "clear_sky":  [("CLOUD_AMT", "low")],
    "wind_dust":  [("WS2M", "high")],
    "low_light":  [("CLRSKY_SFC_SW_DWN", "low")],
}
for f, parts in FACTORS.items():
    cols = [df[c].rank(pct=True, ascending=(d == "high")) for c, d in parts]
    df["f_" + f] = pd.concat(cols, axis=1).mean(axis=1)

TARGETS = {
    "Mars": {"dryness": 3, "cold": 2, "thin_air": 1.5, "uv": 1,
             "temp_swing": 1, "clear_sky": 1, "wind_dust": 0.5},
    "Moon south pole": {"cold_min": 3, "low_light": 2, "dryness": 2,
                        "thin_air": 1, "clear_sky": 1},
}

def score(frame, weights):
    num = sum(frame["f_" + f].fillna(0) * w for f, w in weights.items())
    den = sum(frame["f_" + f].notna() * w for f, w in weights.items())
    return num / den

MON = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for tname, w in TARGETS.items():
    df["s"] = score(df, w)
    best = df.loc[df.groupby("pid")["s"].idxmax()].copy()
    best["best_month"] = best["month"].map(lambda m: MON[m])
    best = best.sort_values("s", ascending=False).reset_index(drop=True)
    best["rank"] = best.index + 1
    print(f"\n===== {tname}: TOP 12 places (best month) =====")
    print(best.head(12)[["rank","kind","name","lat","lon","elev","best_month","s"]].round(3).to_string(index=False))
    print(f"--- {tname}: where the famous benchmark sites rank (out of {len(best)}) ---")
    print(best[best.kind=="known"][["rank","name","best_month","s"]].round(3).to_string(index=False))

In [ ]:
for tname, w in TARGETS.items():
    df["s"] = score(df, w)
    best = df.loc[df.groupby("pid")["s"].idxmax()].copy()
    best = best[(best.kind == "grid") & (best.lat > -60)]   # leave out Antarctica
    best["best_month"] = best["month"].map(lambda m: MON[m])
    print(f"\n===== {tname}: TOP 15 outside Antarctica =====")
    print(best.sort_values("s", ascending=False).head(15)[["lat","lon","elev","best_month","s"]].round(3).to_string(index=False))

In [ ]:
REGIONS = {
    "Puna / Altiplano (Andes)": (-31, -20, -71, -64),
    "Qaidam Basin (China)":     (35, 39, 89, 98),
    "Tibet / Changtang":        (29, 35, 80, 99),
    "Pamir":                    (37, 40, 71, 76),
}
fine = []
for rname, (la0, la1, lo0, lo1) in REGIONS.items():
    for lat in range(la0, la1 + 1):
        for lon in range(lo0, lo1 + 1):
            if globe.is_land(lat, lon):
                fine.append({"kind": "fine", "name": rname, "lat": float(lat), "lon": float(lon)})
print("Fine points:", len(fine))

fine_results, lost = [], []
with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(fetch, p): p for p in fine}
    for i, f in enumerate(as_completed(futures), 1):
        res = f.result()
        if res: fine_results.append(res)
        else: lost.append(futures[f])
        if i % 50 == 0: print(f"{i}/{len(fine)}")
print("OK:", len(fine_results), "| lost:", len(lost))

global_vals = {}
for col in set(c for parts in FACTORS.values() for c, _ in parts):
    v = df.loc[df.kind == "grid", col].dropna().values
    global_vals[col] = np.sort(v)

def pct(col, x, d):
    if x is None or x <= -998: return np.nan
    arr = global_vals[col]
    p = np.searchsorted(arr, x, side="right") / len(arr)
    return p if d == "high" else 1 - p

rows = []
for r in fine_results:
    for mi in range(12):
        row = {"region": r["name"], "lat": r["lat"], "lon": r["lon"], "elev": r["elev"], "month": mi}
        for f, parts in FACTORS.items():
            row["f_" + f] = np.nanmean([pct(c, r["m"][c][mi], d) for c, d in parts])
        rows.append(row)
fdf = pd.DataFrame(rows)

w = TARGETS["Mars"]
fdf["s"] = score(fdf, w)
top = fdf.loc[fdf.groupby(["lat", "lon"])["s"].idxmax()].copy()
top["best_month"] = top["month"].map(lambda m: MON[m])
for rname in REGIONS:
    sub = top[top.region == rname].sort_values("s", ascending=False).head(5)
    print(f"\n=== {rname}: best 5 spots for Mars ===")
    print(sub[["lat", "lon", "elev", "best_month", "s"]].round(3).to_string(index=False))

with open("fine_points.json", "w") as fh:
    json.dump(fine_results, fh)

In [ ]:
FACT_LIST = list(FACTORS.keys())
SHOW = ["T2M", "T2M_MIN", "T2M_MAX", "PRECTOTCORR", "RH2M", "PS",
        "ALLSKY_SFC_UV_INDEX", "CLOUD_AMT", "CLRSKY_SFC_SW_DWN", "WS2M"]

def factor_months(r):
    out = {}
    for f, parts in FACTORS.items():
        vals = []
        for mi in range(12):
            ps = [pct(c, r["m"][c][mi], d) for c, d in parts]
            ps = [p for p in ps if not np.isnan(p)]
            vals.append(round(float(np.mean(ps)), 2) if ps else None)
        out[f] = vals
    return out

def raw_months(r):
    return {c: [None if (v is None or v <= -998) else round(v, 1) for v in r["m"][c]] for c in SHOW}

app = {}
for r in results + fine_results:
    key = (r["lat"], r["lon"])
    if key in app and r["kind"] == "grid":
        continue
    app[key] = {"lat": r["lat"], "lon": r["lon"], "elev": r["elev"],
                "kind": r["kind"], "name": r["name"],
                "f": factor_months(r), "raw": raw_months(r)}

points_out = list(app.values())
with open("app_points.json", "w") as fh:
    json.dump(points_out, fh, separators=(",", ":"))
print("Points:", len(points_out))
import os; print("Size (MB):", round(os.path.getsize("app_points.json") / 1e6, 2))